<a href="https://colab.research.google.com/github/Subuktageen-Farooqi/ms_course_deeplearning/blob/main/ms_deeplearning_tutorial_08b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Object Detection: Faster RCNN Pretrained - Tensorflow Implementation

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import cv2
import matplotlib.pyplot as plt
import os
from PIL import Image

# Step 1: Load the Pre-trained Faster R-CNN Model from TensorFlow Hub
MODEL_URL = "https://tfhub.dev/tensorflow/faster_rcnn/resnet50_v1_640x640/1"
model = hub.load(MODEL_URL)

# Class labels for COCO Dataset (used by the pre-trained model)
COCO_CLASSES = [
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat", "traffic light",
    "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat", "dog", "horse", "sheep", "cow", "elephant",
    "bear", "zebra", "giraffe", "backpack", "umbrella", "handbag", "tie", "suitcase", "frisbee", "skis", "snowboard",
    "sports ball", "kite", "baseball bat", "baseball glove", "skateboard", "surfboard", "tennis racket", "bottle",
    "wine glass", "cup", "fork", "knife", "spoon", "bowl", "banana", "apple", "sandwich", "orange", "broccoli",
    "carrot", "hot dog", "pizza", "donut", "cake", "chair", "couch", "potted plant", "bed", "dining table", "toilet",
    "TV", "laptop", "mouse", "remote", "keyboard", "cell phone", "microwave", "oven", "toaster", "sink", "refrigerator",
    "book", "clock", "vase", "scissors", "teddy bear", "hair drier", "toothbrush"
]

# TensorFlow Hub Faster R-CNN outputs original COCO category IDs (1-90 with gaps).
# Map those sparse IDs to class names to avoid index shifts / IndexError.
COCO_CATEGORY_IDS = [
    1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25,
    27, 28, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 46, 47, 48, 49, 50, 51, 52,
    53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 67, 70, 72, 73, 74, 75, 76, 77, 78, 79,
    80, 81, 82, 84, 85, 86, 87, 88, 89, 90
]
COCO_ID_TO_LABEL = dict(zip(COCO_CATEGORY_IDS, COCO_CLASSES))

# Function to perform object detection
def detect_objects(image_path, model, threshold=0.5):
    # Debug message to verify the file path
    print(f"Loading image from: {image_path}")

    # Load the image
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(f"Could not load image at {image_path}. Please check the file path.")

    # Convert to RGB format for visualization and TensorFlow processing
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Resize image to the model's expected input size
    input_tensor = tf.convert_to_tensor(image_rgb, dtype=tf.uint8)
    input_tensor = input_tensor[tf.newaxis, ...]  # Add batch dimension

    # Run object detection
    detections = model(input_tensor)

    # Extract detection results
    boxes = detections['detection_boxes'][0].numpy()  # Bounding boxes
    class_ids = detections['detection_classes'][0].numpy().astype(int)  # Raw sparse COCO category IDs
    scores = detections['detection_scores'][0].numpy()  # Confidence scores

    # Filter out detections below the confidence threshold
    valid_detections = (scores >= threshold) & np.isin(class_ids, COCO_CATEGORY_IDS)
    boxes = boxes[valid_detections]
    class_labels = np.array([COCO_ID_TO_LABEL[class_id] for class_id in class_ids[valid_detections]])
    scores = scores[valid_detections]

    return image_rgb, boxes, class_labels, scores

# Function to display the image with detected bounding boxes
def display_image_with_detections(image, boxes, class_labels, scores, threshold=0.5):
    height, width, _ = image.shape
    for i, box in enumerate(boxes):
        if scores[i] >= threshold:
            ymin, xmin, ymax, xmax = box
            (left, right, top, bottom) = (int(xmin * width), int(xmax * width), int(ymin * height), int(ymax * height))

            # Draw bounding box
            cv2.rectangle(image, (left, top), (right, bottom), (0, 255, 0), 2)

            # Display label and confidence
            label = f"{class_labels[i]}: {scores[i]:.2f}"
            cv2.putText(image, label, (left, top - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Show the result with matplotlib
    plt.figure(figsize=(12, 8))
    plt.imshow(image)
    plt.axis('off')
    plt.show()

# Example usage with your image path
image_path = "D:\\Deep Learning Course\\Codes\\Images for pretrained\\Sample image.jpg"
try:
    image_rgb, detected_boxes, detected_classes, detected_scores = detect_objects(image_path, model, threshold=0.5)
    display_image_with_detections(image_rgb, detected_boxes, detected_classes, detected_scores)
except FileNotFoundError as e:
    print(e)

# Task 1: PyTorch Implementation

In [ ]:
# Task 1: PyTorch implementation of the Faster R-CNN tutorial workflow.
import torch
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights


def load_model(device):
    weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
    model = fasterrcnn_resnet50_fpn(weights=weights)
    model = model.to(device)
    model.eval()
    categories = weights.meta.get("categories", [])
    preprocess = weights.transforms()
    return model, preprocess, categories


def load_image(image_path):
    image = Image.open(image_path).convert("RGB")
    return image


def detect_objects_pytorch(image, model, preprocess, device, score_threshold=0.5):
    input_tensor = preprocess(image).to(device)
    with torch.inference_mode():
        output = model([input_tensor])[0]

    scores = output["scores"]
    keep = scores >= score_threshold
    filtered = {
        "boxes": output["boxes"][keep].cpu(),
        "labels": output["labels"][keep].cpu(),
        "scores": output["scores"][keep].cpu(),
    }
    return filtered


def draw_detections(image, detections, categories):
    fig, ax = plt.subplots(1, figsize=(12, 8))
    ax.imshow(image)

    for box, label, score in zip(detections["boxes"], detections["labels"], detections["scores"]):
        x1, y1, x2, y2 = box.tolist()
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor="lime", facecolor="none")
        ax.add_patch(rect)

        label_idx = int(label.item())
        class_name = categories[label_idx] if 0 <= label_idx < len(categories) else f"class_{label_idx}"
        ax.text(x1, max(0, y1 - 5), f"{class_name}: {score.item():.2f}", color="lime", fontsize=9, backgroundcolor="black")

    ax.axis("off")
    plt.tight_layout()
    plt.show()


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
image_path = "sample_image.jpg"
score_threshold = 0.5

model, preprocess, categories = load_model(device)
image = load_image(image_path)
detections = detect_objects_pytorch(image, model, preprocess, device, score_threshold=score_threshold)
draw_detections(image, detections, categories)

Task 02: Take another two models with different backbones

In [ ]:
# Task 2: Inference comparison using two other pretrained detection backbones/families.
import time
from pathlib import Path

import torch
from PIL import Image
import matplotlib.pyplot as plt
from torchvision.models.detection import (
    fasterrcnn_mobilenet_v3_large_fpn,
    FasterRCNN_MobileNet_V3_Large_FPN_Weights,
    retinanet_resnet50_fpn_v2,
    RetinaNet_ResNet50_FPN_V2_Weights,
)
from torchvision.transforms.functional import pil_to_tensor
from torchvision.utils import draw_bounding_boxes


def load_image(image_path):
    return Image.open(image_path).convert("RGB")


def get_model_registry(device):
    return {
        "fasterrcnn_mobilenet_v3_large_fpn": {
            "weights": FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT,
            "builder": fasterrcnn_mobilenet_v3_large_fpn,
            "device": device,
        },
        "retinanet_resnet50_fpn_v2": {
            "weights": RetinaNet_ResNet50_FPN_V2_Weights.DEFAULT,
            "builder": retinanet_resnet50_fpn_v2,
            "device": device,
        },
    }


def run_inference_for_model(model_name, config, image, score_threshold=0.5, top_k_preview=5):
    weights = config["weights"]
    model = config["builder"](weights=weights).to(config["device"])
    model.eval()
    preprocess = weights.transforms()
    categories = weights.meta.get("categories", [])

    input_tensor = preprocess(image).to(config["device"])
    start = time.perf_counter()
    with torch.inference_mode():
        output = model([input_tensor])[0]
    elapsed = time.perf_counter() - start

    keep = output["scores"] >= score_threshold
    boxes = output["boxes"][keep].detach().cpu()
    labels = output["labels"][keep].detach().cpu()
    scores = output["scores"][keep].detach().cpu()

    preview = []
    for label, score in zip(labels[:top_k_preview], scores[:top_k_preview]):
        idx = int(label.item())
        name = categories[idx] if 0 <= idx < len(categories) else f"class_{idx}"
        preview.append(f"{name}:{score.item():.2f}")

    return {
        "model_name": model_name,
        "boxes": boxes,
        "labels": labels,
        "scores": scores,
        "categories": categories,
        "num_detections": int(keep.sum().item()),
        "preview": preview,
        "inference_time_s": elapsed,
    }


def save_and_display_predictions(image, result, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    image_tensor = pil_to_tensor(image)

    box_labels = []
    for label, score in zip(result["labels"], result["scores"]):
        idx = int(label.item())
        name = result["categories"][idx] if 0 <= idx < len(result["categories"]) else f"class_{idx}"
        box_labels.append(f"{name}:{score.item():.2f}")

    drawn = draw_bounding_boxes(image_tensor, result["boxes"].to(torch.int64), labels=box_labels, width=2, colors="green")
    drawn_np = drawn.permute(1, 2, 0).numpy()

    output_path = output_dir / f"{result['model_name']}_predictions.jpg"
    Image.fromarray(drawn_np).save(output_path)

    plt.figure(figsize=(12, 8))
    plt.title(result["model_name"])
    plt.imshow(drawn_np)
    plt.axis("off")
    plt.tight_layout()
    plt.show()


def compare_models_on_image(image_path, score_threshold=0.5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    image = load_image(image_path)
    registry = get_model_registry(device)
    output_dir = Path("task2_outputs")

    print(f"Comparing models on: {image_path}")
    print(f"Score threshold: {score_threshold}")
    print(f"Device: {device}")

    for model_name, config in registry.items():
        result = run_inference_for_model(model_name, config, image, score_threshold=score_threshold)
        save_and_display_predictions(image, result, output_dir)

        print(f"\nModel: {result['model_name']}")
        print(f"Detections >= threshold: {result['num_detections']}")
        print(f"Top labels: {', '.join(result['preview']) if result['preview'] else 'None'}")
        print(f"Approx inference time: {result['inference_time_s']:.4f}s")


image_path = "sample_image.jpg"
score_threshold = 0.5
compare_models_on_image(image_path, score_threshold=score_threshold)

Task 03: Using your own dataset; evaluate and then find its mAP

In [ ]:
# Task 3: Dataset-level mAP evaluation for a pretrained object detector (COCO-format).
from pathlib import Path
import json

import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights

try:
    from torchmetrics.detection.mean_ap import MeanAveragePrecision
except Exception:
    MeanAveragePrecision = None


DATASET_ROOT = Path("dataset")
IMAGES_DIR = DATASET_ROOT / "images"
ANNOTATIONS_FILE = DATASET_ROOT / "annotations" / "instances.json"
BATCH_SIZE = 1
SCORE_THRESHOLD = 0.05


class COCODetectionDataset(Dataset):
    def __init__(self, images_dir, coco, transforms, category_id_map):
        self.images_dir = Path(images_dir)
        self.transforms = transforms
        self.category_id_map = category_id_map

        self.images = sorted(coco["images"], key=lambda x: x["id"])

        ann_by_image = {}
        for ann in coco["annotations"]:
            ann_by_image.setdefault(ann["image_id"], []).append(ann)
        self.ann_by_image = ann_by_image

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image_info = self.images[idx]
        image_path = self.images_dir / image_info["file_name"]
        image = Image.open(image_path).convert("RGB")

        anns = self.ann_by_image.get(image_info["id"], [])
        boxes = []
        labels = []
        areas = []
        iscrowd = []

        for ann in anns:
            x, y, w, h = ann["bbox"]
            boxes.append([x, y, x + w, y + h])
            raw_category_id = ann["category_id"]
            if raw_category_id not in self.category_id_map:
                raise KeyError(f"Unknown category_id {raw_category_id} in annotation for image_id={image_info['id']}")
            labels.append(self.category_id_map[raw_category_id])
            areas.append(ann.get("area", w * h))
            iscrowd.append(ann.get("iscrowd", 0))

        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0, 4), dtype=torch.float32),
            "labels": torch.tensor(labels, dtype=torch.int64) if labels else torch.zeros((0,), dtype=torch.int64),
            "image_id": torch.tensor([image_info["id"]], dtype=torch.int64),
            "area": torch.tensor(areas, dtype=torch.float32) if areas else torch.zeros((0,), dtype=torch.float32),
            "iscrowd": torch.tensor(iscrowd, dtype=torch.int64) if iscrowd else torch.zeros((0,), dtype=torch.int64),
        }

        image_tensor = self.transforms(image)
        return image_tensor, target


def build_category_id_map(coco_categories, model_categories):
    dataset_id_to_name = {cat["id"]: cat["name"].strip().lower() for cat in coco_categories if "id" in cat and "name" in cat}

    model_name_to_label = {
        name.strip().lower(): idx
        for idx, name in enumerate(model_categories)
        if name and name != "N/A"
    }

    if dataset_id_to_name and model_name_to_label:
        missing_names = sorted({name for name in dataset_id_to_name.values() if name not in model_name_to_label})
        if missing_names:
            missing_preview = ", ".join(missing_names[:5])
            raise ValueError(
                "Dataset category names are not aligned with model labels. "
                f"Missing matches for: {missing_preview}"
            )

        return {dataset_id: model_name_to_label[name] for dataset_id, name in dataset_id_to_name.items()}

    dataset_ids = {cat["id"] for cat in coco_categories if "id" in cat}
    model_valid_ids = {
        idx for idx, name in enumerate(model_categories) if name and name != "N/A"
    }

    if dataset_ids and dataset_ids.issubset(model_valid_ids):
        return {dataset_id: dataset_id for dataset_id in dataset_ids}

    raise ValueError(
        "Unable to build category-id mapping from dataset categories to model label space. "
        "Provide COCO categories with names that match the model's category names."
    )


def collate_fn(batch):
    images, targets = zip(*batch)
    return list(images), list(targets)


def evaluate_map(dataset_root=DATASET_ROOT, score_threshold=SCORE_THRESHOLD):
    dataset_root = Path(dataset_root)
    images_dir = dataset_root / "images"
    annotations_file = dataset_root / "annotations" / "instances.json"

    if MeanAveragePrecision is None:
        print("torchmetrics is not available. Install torchmetrics to compute mAP/mAP50/mAP75.")
        return

    if not images_dir.exists() or not annotations_file.exists():
        raise FileNotFoundError(
            "Expected COCO-format dataset at dataset/images and dataset/annotations/instances.json"
        )

    with open(annotations_file, "r", encoding="utf-8") as f:
        coco = json.load(f)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
    model = fasterrcnn_resnet50_fpn(weights=weights).to(device)
    model.eval()

    category_id_map = build_category_id_map(
        coco_categories=coco.get("categories", []),
        model_categories=weights.meta.get("categories", []),
    )

    dataset = COCODetectionDataset(
        images_dir=images_dir,
        coco=coco,
        transforms=weights.transforms(),
        category_id_map=category_id_map,
    )
    data_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

    metric = MeanAveragePrecision(iou_type="bbox")

    with torch.inference_mode():
        for images, targets in data_loader:
            images = [img.to(device) for img in images]
            outputs = model(images)

            preds = []
            for out in outputs:
                keep = out["scores"] >= score_threshold
                preds.append(
                    {
                        "boxes": out["boxes"][keep].detach().cpu(),
                        "scores": out["scores"][keep].detach().cpu(),
                        "labels": out["labels"][keep].detach().cpu(),
                    }
                )

            target_list = []
            for tgt in targets:
                target_list.append(
                    {
                        "boxes": tgt["boxes"].detach().cpu(),
                        "labels": tgt["labels"].detach().cpu(),
                    }
                )

            metric.update(preds, target_list)

    results = metric.compute()
    model_name = "fasterrcnn_resnet50_fpn"
    split_name = "dataset/images + dataset/annotations/instances.json"

    print(f"Model: {model_name}")
    print(f"Category ID mapping entries: {len(category_id_map)}")
    print(f"Evaluated split: {split_name}")
    print(f"mAP: {results['map'].item():.4f}")
    print(f"mAP50: {results['map_50'].item():.4f}")
    if "map_75" in results:
        print(f"mAP75: {results['map_75'].item():.4f}")


evaluate_map()